# NLP Lab: Text Preprocessing and Spelling Correction

Welcome to this Natural Language Processing (NLP) lab. In this session, we will explore fundamental text preprocessing techniques that serve as the building blocks for most NLP pipelines.

Topics Covered:
1. Regular Expressions (Regex) for text cleaning
2. Tokenization and Stop Word Removal
3. Stemming (Porter's Algorithm)
4. Lemmatization
5. Edit Distance (Levenshtein) for Spelling Correction

Let's start by importing the necessary libraries and downloading the required NLTK datasets.

In [ ]:
%pip install nltk

In [7]:
import re
import nltk

# Download required NLTK data (run this once)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("Libraries imported and NLTK data downloaded successfully!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Libraries imported and NLTK data downloaded successfully!


## 1. Regular Expressions (Regex)
Regular expressions are powerful tools for pattern matching and text manipulation. In NLP, we frequently use them to clean raw text by removing HTML tags, special characters, numbers, and extra whitespaces.

In [8]:
# Sample messy text
raw_text = "Hello!!! Welcome to the NLP lab in 2026. This text has <br> HTML tags, numbers like 123, and symbols #$%^."

# 1. Remove HTML tags
no_html = re.sub(r'<[^>]+>', '', raw_text)

# 2. Remove special characters and numbers (keep only letters and spaces)
cleaned_text = re.sub(r'[^A-Za-z\s]', '', no_html)

# 3. Remove extra whitespaces
cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

print("Original Text:", raw_text)
print("Cleaned Text :", cleaned_text)

Original Text: Hello!!! Welcome to the NLP lab in 2026. This text has <br> HTML tags, numbers like 123, and symbols #$%^.
Cleaned Text : Hello Welcome to the NLP lab in This text has HTML tags numbers like and symbols


## 2. Tokenization and Stop Word Removal
Tokenization is the process of breaking text down into smaller units, such as words.
Stop words are highly common words (e.g., "the", "is", "in", "and") that often do not carry significant semantic meaning for tasks like classification or topic modeling. Removing them reduces the size of our dataset and helps models focus on meaningful words.

In [9]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Tokenize the cleaned text
tokens = word_tokenize(cleaned_text.lower())

# Load English stop words
stop_words = set(stopwords.words('english'))

# Remove stop words
filtered_tokens = [word for word in tokens if word not in stop_words]

print("Tokens before stop word removal:", tokens)
print("Tokens after stop word removal :", filtered_tokens)

Tokens before stop word removal: ['hello', 'welcome', 'to', 'the', 'nlp', 'lab', 'in', 'this', 'text', 'has', 'html', 'tags', 'numbers', 'like', 'and', 'symbols']
Tokens after stop word removal : ['hello', 'welcome', 'nlp', 'lab', 'text', 'html', 'tags', 'numbers', 'like', 'symbols']


## 3. Stemming (Porter's Algorithm)
Stemming is a crude heuristic process that chops off the ends of words to reduce them to their base or root form.
The Porter Stemming Algorithm is one of the oldest and most popular stemming algorithms. It uses a predefined set of rules to strip suffixes.

*Note: Stemming can sometimes result in non-words (e.g., "universe" might become "univers").*

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

# Words to stem
words_to_stem = ["running", "runs", "ran", "runner", "easily", "fairly", "universe"]

# Apply Porter Stemmer
stemmed_words = [stemmer.stem(word) for word in words_to_stem]

print("Original Words :", words_to_stem)
print("Stemmed Words  :", stemmed_words)

Original Words : ['running', 'runs', 'ran', 'runner', 'easily', 'fairly', 'universe']
Stemmed Words  : ['run', 'run', 'ran', 'runner', 'easili', 'fairli', 'univers']


## 4. Lemmatization
Unlike stemming, Lemmatization utilizes vocabulary and morphological analysis to return the dictionary form of a word, known as the lemma. It is computationally more expensive than stemming but produces actual words.

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

# We need to specify the Part of Speech (POS) tag for accurate lemmatization.
# 'v' stands for verb, 'a' for adjective, 'n' for noun.
print("--- Comparing Stemming vs Lemmatization ---")
for word in words_to_stem:
    stemmed = stemmer.stem(word)
    # Default POS is noun ('n'), let's try verb ('v') for our list
    lemmatized = lemmatizer.lemmatize(word, pos='v')
    print(f"Original: {word:10} | Stemmed: {stemmed:10} | Lemmatized: {lemmatized}")

--- Comparing Stemming vs Lemmatization ---


LookupError: 
**********************************************************************
  Resource [93mwordnet[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('wordnet')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mcorpora/wordnet[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


## 5. Edit Distance (Levenshtein Distance)
The Edit Distance algorithm measures how dissimilar two strings are by counting the minimum number of operations required to transform one string into the other.

The allowable operations are:
1. Insertion of a character
2. Deletion of a character
3. Substitution of a character

This is widely used in spell checkers. Let's write a dynamic programming function to calculate the Edit Distance, and then build a simple spelling corrector.

In [11]:
import numpy as np

def calculate_edit_distance(word1, word2):
    m, n = len(word1), len(word2)

    dp = np.zeros((m + 1, n + 1), dtype=int)

    # Initialize first column
    for i in range(m + 1):
        dp[i][0] = i

    # Initialize first row
    for j in range(n + 1):
        dp[0][j] = j

    # Fill DP table
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost # substitution
            )

    return dp[m][n]

a = "Kitten"
b = "Sitting"

dist = calculate_edit_distance(a, b)
print(dist)

3


### 5.1 Spell Checking / Autocorrect Application
Now, let's use our Edit Distance function to correct a misspelled word by comparing it against a known "dictionary" (vocabulary list). We will suggest the word with the lowest edit distance.

In [ ]:
# A tiny dictionary of valid words
vocabulary = ["apple", "banana", "orange", "grape", "strawberry", "pineapple", "mango"]

# A misspelled word
misspelled_word = "bananna"

def correct_spelling(word, vocab):
    # Calculate distance from the misspelled word to every word in the dictionary
    distances = {valid_word: calculate_edit_distance(word, valid_word) for valid_word in vocab}

    # Sort the dictionary by the distance (lowest distance first)
    sorted_distances = sorted(distances.items(), key=lambda item: item[1])

    print(f"Distances evaluated for '{word}': {sorted_distances[:3]} ...")

    # The best suggestion is the one with the minimum edit distance
    best_suggestion = sorted_distances[0][0]
    return best_suggestion

suggestion = correct_spelling(misspelled_word, vocabulary)
print(f"\nDid you mean: '{suggestion}' instead of '{misspelled_word}'?")

NameError: name 'calculate_edit_distance' is not defined